# Montpelier VT 2023 Flood — Full IN-CORE Pipeline

End-to-end workflow:
1. **Flood depth raster** — from NOAA FIM library for MONV1 (Winooski at Montpelier)
2. **Building inventory** — from National Structure Inventory (NSI) POST API
3. **Building damage** — pyIncore `BuildingStructuralDamage` with Lumberton flood fragilities
4. **Monte Carlo failure probability** — pyIncore `MonteCarloFailureProbability`
5. **Housing unit allocation** — pyIncore `HousingUnitAllocation` (requires Census + address data)
6. **Population dislocation** — pyIncore `PopulationDislocation`

**Python interpreter:** `/opt/anaconda3/envs/hecras/bin/python`

**Prerequisites:**
```bash
# Create env (one time)
/opt/anaconda3/bin/conda create -n hecras python=3.10 "numpy<2" -y

# Install dependencies
/opt/anaconda3/envs/hecras/bin/pip install \
    dataretrieval geopandas rasterio requests scipy \
    pyincore pyincore-data
```

**IN-CORE account required for Part 3** (free at https://tools.in-core.org)

## 0. Imports and configuration

In [1]:
!pip install dataretrieval

In [2]:
import os, uuid, json, warnings
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.crs import CRS
from shapely.geometry import box, mapping

warnings.filterwarnings('ignore')

# ── Spatial config ────────────────────────────────────────────────────────────
# Downtown Montpelier floodplain bounding box (lon_min, lat_min, lon_max, lat_max)
# Covers the Winooski / North Branch confluence and State Street area
BBOX = (-72.590, 44.250, -72.550, 44.270)

# NOAA FIM library for MONV1 (Winooski River at Montpelier)
# Peak stage July 11 2023: 21.29 ft  |  Gage datum: 499.87 ft NAVD88
# Peak WSE: 521.16 ft  →  closest pre-computed grid: elev_521_6 (521.6 ft)
NOAA_FIM_ZIP = "https://water.noaa.gov/resources/downloads/fim/btv/monv1/shapefile/monv1_shapefiles.zip"
FIM_GRID     = "noaa_fim/shapefiles/shp/ahps/inundation/monv1/depth_grids/elev_521_6"

# USGS gage: Winooski River at Montpelier
GAGE_ID = "04286000"

OUT_DIR  = "./outputs"
FIM_DIR  = "./noaa_fim"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIM_DIR, exist_ok=True)

FLOOD_RASTER = f"{OUT_DIR}/montpelier_flood_depth_2023.tif"
BLDG_SHP     = f"{OUT_DIR}/montpelier_building_inventory/montpelier_building_inventory.shp"

print("Config OK")

Config OK


---
## Part 1 — Flood Depth Raster

Source: NOAA FIM library for AHPS gauge **MONV1** (Winooski River at Montpelier).  
NOAA publishes pre-computed depth grids for a range of water surface elevations.  
We select the grid closest to the July 2023 event peak.

**Skip this part if `outputs/montpelier_flood_depth_2023.tif` already exists.**

### 1.1 Verify July 2023 peak flow at USGS gage

In [3]:
import dataretrieval.nwis as nwis

peaks_df, _ = nwis.get_discharge_peaks(sites=GAGE_ID)
peaks_df.index = pd.to_datetime(peaks_df.index, utc=True)
flood_peak = peaks_df[peaks_df.index.to_series().between("2023-07-01", "2023-07-31")]

print("July 2023 peak at Winooski River at Montpelier (USGS 04286000):")
print(flood_peak[["peak_va", "gage_ht"]])
# Expected: peak_va ~23100 cfs, gage_ht 21.29 ft on 2023-07-11
# Gage datum 499.87 ft NAVD88  →  WSE = 21.29 + 499.87 = 521.16 ft

July 2023 peak at Winooski River at Montpelier (USGS 04286000):
                           peak_va  gage_ht
datetime                                   
2023-07-11 00:00:00+00:00    23100    21.29


### 1.2 Download NOAA FIM library for MONV1

In [4]:
import zipfile

zip_path = f"{FIM_DIR}/monv1_shapefiles.zip"

if os.path.exists(FIM_GRID):
    print(f"FIM grid already extracted: {FIM_GRID}")
else:
    if not os.path.exists(zip_path):
        print("Downloading NOAA FIM library (~14 MB) ...")
        resp = requests.get(NOAA_FIM_ZIP, stream=True, timeout=120)
        resp.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {os.path.getsize(zip_path)/1e6:.1f} MB")
    else:
        print(f"Zip cached: {zip_path}")

    print("Extracting ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(f"{FIM_DIR}/shapefiles")
    print("Extracted")

# Available depth grids (water surface elevation in ft NAVD88):
import glob
grids = sorted(glob.glob(f"{FIM_DIR}/shapefiles/shp/ahps/inundation/monv1/depth_grids/elev_*"))
grid_names = [os.path.basename(g) for g in grids if os.path.isdir(g)]
print(f"\nAvailable grids: {grid_names}")
print(f"Using: elev_521_6 (WSE 521.6 ft, actual peak 521.16 ft)")

FIM grid already extracted: noaa_fim/shapefiles/shp/ahps/inundation/monv1/depth_grids/elev_521_6

Available grids: ['elev_515_2', 'elev_516_4', 'elev_517_6', 'elev_518_5', 'elev_519_4', 'elev_520_4', 'elev_521_6', 'elev_522_4', 'elev_523_4', 'elev_524_5', 'elev_525_7', 'elev_526_6', 'elev_527_5', 'elev_528_4']
Using: elev_521_6 (WSE 521.6 ft, actual peak 521.16 ft)


### 1.3 Read depth grid, clip to downtown, save GeoTIFF

In [5]:
# The NOAA FIM grids are Esri GRID format (ADF) in EPSG:3857 (Web Mercator)
crs_3857 = CRS.from_epsg(3857)

with rasterio.open(FIM_GRID) as src:
    arr = src.read(1).astype(np.float32)
    arr[arr == src.nodata] = np.nan   # mask no-data
    transform = src.transform

# Write as GeoTIFF with CRS (AIG driver can't write, so we convert)
tmp = f"{OUT_DIR}/tmp_full.tif"
meta = {"driver": "GTiff", "dtype": "float32", "nodata": np.nan,
        "width": arr.shape[1], "height": arr.shape[0],
        "count": 1, "crs": crs_3857, "transform": transform}
with rasterio.open(tmp, "w", **meta) as dst:
    dst.write(arr[np.newaxis, :, :])

# Clip to downtown bounding box
bbox_gdf = gpd.GeoDataFrame(geometry=[box(*BBOX)], crs="EPSG:4326").to_crs(crs_3857)
with rasterio.open(tmp) as src:
    clipped, clip_tf = rio_mask(src, bbox_gdf.geometry, crop=True, nodata=np.nan)
    clip_meta = src.meta.copy()
    clip_meta.update({"height": clipped.shape[1], "width": clipped.shape[2],
                      "transform": clip_tf})

with rasterio.open(FLOOD_RASTER, "w", **clip_meta) as dst:
    dst.write(clipped)
os.remove(tmp)

c = clipped[0]
print(f"Flood raster saved : {FLOOD_RASTER}")
print(f"Shape              : {c.shape}")
print(f"Max depth downtown : {np.nanmax(c):.2f} ft above ground")
print(f"Mean depth (wet)   : {np.nanmean(c[c > 0]):.2f} ft")
print(f"Inundated cells    : {(c > 0).sum():,}")

Flood raster saved : ./outputs/montpelier_flood_depth_2023.tif
Shape              : (1227, 1827)
Max depth downtown : 23.78 ft above ground
Mean depth (wet)   : 6.77 ft
Inundated cells    : 202,219


---
## Part 2 — Building Inventory from NSI

The NSI GET endpoint returns 500 for some bounding boxes; the POST endpoint with a GeoJSON polygon body is reliable.

**Skip this part if `outputs/montpelier_building_inventory/` already exists.**

### 2.1 Query NSI API (POST)

In [6]:
payload = {"type": "Feature", "geometry": mapping(box(*BBOX)), "properties": {}}
resp = requests.post(
    "https://nsi.sec.usace.army.mil/nsiapi/structures?fmt=fc",
    json=payload,
    headers={"Content-Type": "application/json"},
    timeout=60
)
resp.raise_for_status()
nsi_gdf = gpd.GeoDataFrame.from_features(resp.json()["features"], crs="EPSG:4326")
print(f"Buildings fetched: {len(nsi_gdf)}")
nsi_gdf.head(3)

Buildings fetched: 2131


,geometry,fd_id,bid,occtype,st_damcat,bldgtype,found_type,cbfips,pop2amu65,pop2amo65,...,val_vehic,source,med_yr_blt,firmzone,o65disable,u65disable,x,y,ground_elv,ground_elv_m
0,POINT (-72.5711 44.2513),582073589,87P97C2H+GHC-0-0-0-0,RES1-3SNB,RES,W,S,500239549002006,2,0,...,27000,X,1958,None,0.16,0.03,-72.571098,44.251305,534.159510,162.811813
1,POINT (-72.58051 44.26248),582001207,87P97C69+XQX-9-12-8-12,GOV1,PUB,S,S,500239546002004,1,0,...,2790000,E,1939,None,0.16,0.03,-72.580506,44.262485,552.542847,168.415054
2,POINT (-72.58051 44.26248),582001208,87P97C69+XQX-9-12-8-12,COM4,COM,S,S,500239546002004,0,0,...,2331000,E,1939,None,0.16,0.03,-72.580506,44.262485,552.542847,168.415054


### 2.2 Map NSI occupancy types → IN-CORE archetypes (Nofal & van de Lindt 2020)

In [7]:
ARCHETYPE_MAP = {
    # 1-story residential
    "RES1-1S": 1, "RES1-1SNB": 1, "RES1-1SWB": 1,
    # 2-story residential
    "RES1-2S": 3, "RES1-2SNB": 3, "RES1-2SWB": 3,
    # 3-story single-family → treated as 2-story archetype
    "RES1-3S": 3, "RES1-3SNB": 3, "RES1-3SWB": 3,
    "RES1-SL": 5,   # split-level
    "RES2": 6,      # mobile home
    "RES3A": 12, "RES3B": 12, "RES3C": 12,
    "RES3D": 12, "RES3E": 12, "RES3F": 12,  # multifamily
    "RES4": 12, "RES5": 13, "RES6": 13,
    "COM1": 7,  "COM2": 8,  "COM3": 7,  "COM4": 9,  "COM5": 9,
    "COM6": 15, "COM7": 10, "COM8": 13, "COM9": 13, "COM10": 13,
    "IND1": 11, "IND2": 11, "IND3": 11, "IND4": 11, "IND5": 11, "IND6": 11,
    "AGR1": 11, "REL1": 13, "GOV1": 10, "GOV2": 15, "EDU1": 14, "EDU2": 14,
}

def map_archetype(occ):
    if occ in ARCHETYPE_MAP: return ARCHETYPE_MAP[occ]
    return ARCHETYPE_MAP.get(occ.split("-")[0] if "-" in occ else occ, None)

nsi_gdf["archetype"] = nsi_gdf["occtype"].apply(map_archetype)
print("Archetype distribution:")
print(nsi_gdf["archetype"].value_counts(dropna=False).sort_index())

Archetype distribution:
archetype
1     420
3     865
6       1
7     116
8      32
9     119
10     81
11     41
12    345
13     89
14     12
15     10
Name: count, dtype: int64


### 2.3 Rename to IN-CORE Lumberton schema and save

In [8]:
col_map = {
    "fd_id": "strctid", "ground_elv": "g_elev", "num_story": "no_stories",
    "occtype": "occ_type", "val_struct": "repl_cst", "val_cont": "cont_val",
    "sqft": "sq_foot", "bldgtype": "struct_typ",
}
bldg_gdf = nsi_gdf.rename(columns=col_map).copy()
bldg_gdf["ffe_elev"] = (
    bldg_gdf["g_elev"] + nsi_gdf["ffh"] if "ffh" in nsi_gdf.columns
    else bldg_gdf.get("ffe_elev", bldg_gdf["g_elev"])
)
bldg_gdf["guid"] = [str(uuid.uuid4()) for _ in range(len(bldg_gdf))]

bldg_out = f"{OUT_DIR}/montpelier_building_inventory.shp"
bldg_gdf.to_file(bldg_out, driver="ESRI Shapefile")
print(f"Saved: {bldg_out}/  ({len(bldg_gdf)} buildings)")

Saved: ./outputs/montpelier_building_inventory.shp/  (2131 buildings)


---
## Part 3 — pyIncore Analyses

Requires an IN-CORE account (free at https://tools.in-core.org).  
Fragility functions and mapping are fetched from the IN-CORE server.  
The hazard and building inventory are loaded **locally** — no upload needed.

### 3.0 Connect to IN-CORE

In [9]:
from pyincore import IncoreClient, Dataset, FragilityService, MappingSet, DataService
from pyincore.analyses.buildingstructuraldamage import BuildingStructuralDamage
from pyincore.analyses.montecarlolimitstateprobability import MonteCarloLimitStateProbability
from pyincore.analyses.populationdislocation import PopulationDislocation
from pyincore.analyses.housingunitallocation import HousingUnitAllocation
from pyincore.models.hazard.flood import Flood

client = IncoreClient()   # prompts for IN-CORE credentials on first run
client.clear_cache()
data_service = DataService(client)
fragility_service = FragilityService(client)
print("Connected to IN-CORE")

Connection successful to IN-CORE services. pyIncore version detected: 1.22.0
Connected to IN-CORE


### 3.1 Create local flood hazard object

pyIncore supports local (file-based) hazards — no upload required.
The `Flood` object wraps the GeoTIFF and handles spatial interpolation at building locations.

In [10]:
from pyincore.models.hazard.flood import Flood

# Define directly using a dictionary and attach the local TIF
flood_metadata = {
    "type": "flood",
    "name": "Montpelier Flood 2023",
    "description": "Local deterministic flood depth raster",
    "hazardDatasets": [
        {
            "hazardType": "deterministic",
            "demandType": "floodDepth",
            "demandUnits": "ft"
        }
    ]
}
flood = Flood(flood_metadata)
flood.hazardDatasets[0].from_file(FLOOD_RASTER, data_type="incore:floodRaster")

print(f"Local flood hazard loaded: {FLOOD_RASTER}")

Local flood hazard loaded: ./outputs/montpelier_flood_depth_2023.tif


### 3.2 Load building inventory and fragility mapping

In [11]:
from pyincore import Dataset, MappingSet

# Directly read the shp file with guids that was perfectly generated in Section 2.3
local_bldg_dataset = Dataset.from_file(
    "outputs/montpelier_building_inventory.shp", 
    data_type="ergo:buildingInventoryVer7"
)

# Fetch the fragility mapping from the IN-CORE cloud
mapping_id = "602f3cf981bd2c09ad8f4f9d"
mapping_set = MappingSet(fragility_service.get_mapping(mapping_id))

print("Building inventory and fragility mapping loaded successfully.")

Building inventory and fragility mapping loaded successfully.


In [12]:
nsi_gdf.head()

,geometry,fd_id,bid,occtype,st_damcat,bldgtype,found_type,cbfips,pop2amu65,pop2amo65,...,source,med_yr_blt,firmzone,o65disable,u65disable,x,y,ground_elv,ground_elv_m,archetype
0,POINT (-72.5711 44.2513),582073589,87P97C2H+GHC-0-0-0-0,RES1-3SNB,RES,W,S,500239549002006,2,0,...,X,1958,None,0.16,0.03,-72.571098,44.251305,534.159510,162.811813,3
1,POINT (-72.58051 44.26248),582001207,87P97C69+XQX-9-12-8-12,GOV1,PUB,S,S,500239546002004,1,0,...,E,1939,None,0.16,0.03,-72.580506,44.262485,552.542847,168.415054,10
2,POINT (-72.58051 44.26248),582001208,87P97C69+XQX-9-12-8-12,COM4,COM,S,S,500239546002004,0,0,...,E,1939,None,0.16,0.03,-72.580506,44.262485,552.542847,168.415054,9
3,POINT (-72.58051 44.26248),582001209,87P97C69+XQX-9-12-8-12,COM8,COM,S,S,500239546002004,0,0,...,E,1939,None,0.16,0.03,-72.580506,44.262485,552.542847,168.415054,13
4,POINT (-72.5779 44.26215),582001210,87P97C6C+VR9-1-1-2-2,COM4,COM,W,B,500239546002004,0,0,...,E,1939,None,0.16,0.03,-72.577901,44.262152,545.140932,166.158951,9


### 3.3 Run building structural damage analysis

### 3.4 Explore building damage results

In [13]:
import json
from pyincore.models.hazard.flood import Flood


# ---------------- Test reading hazard data (corresponds to the last step in your screenshot) ----------------
# Let's pick a random coordinate point within the downtown BBOX to test the flood depth
payload = [
    {
        "demands": ["floodDepth"],
        "units": ["ft"],
        "loc": "-72.5711,44.2513"
    }
]
values = flood.read_hazard_values(payload)
print("\nTesting flood depth reading at the specified coordinates (if not 0, loading is completely successful):")
print(json.dumps(values, indent=4))


Testing flood depth reading at the specified coordinates (if not 0, loading is completely successful):
[
    {
        "demands": [
            "floodDepth"
        ],
        "units": [
            "ft"
        ],
        "loc": "-72.5711,44.2513",
        "hazardValues": [
            null
        ]
    }
]


In [14]:
from pyincore import Dataset
from pyincore.models.hazard.flood import Flood
from pyincore.analyses.buildingstructuraldamage import BuildingStructuralDamage

print("1. Temporarily uploading local flood raster to your IN-CORE personal space (this only takes a few seconds)...")

dataset_properties = {
    "title": "Montpelier Flood 2023 Temp",
    "dataType": "incore:floodRaster",
    "format": "geotiff"
}

# Pass the property dictionary and the path to the TIF file you want to upload
uploaded_flood_ds = data_service.create_dataset(
    properties=dataset_properties, 
    files=[FLOOD_RASTER]
)

# [Critical Fix]: Native dictionary object, extract using ["id"]!
dataset_id = uploaded_flood_ds["id"]
print(f"Upload successful! Cloud Dataset ID: {dataset_id}")

print("2. Reconstructing the hazard object using the cloud ID...")
flood_metadata = {
    "type": "flood",
    "name": "Montpelier Flood 2023",
    "description": "Uploaded for multiprocessing bypass",
    "hazardDatasets": [
        {
            "hazardType": "deterministic",
            "demandType": "floodDepth",
            "demandUnits": "ft",
            "datasetId": dataset_id  
        }
    ]
}
cloud_flood = Flood(flood_metadata)

print("3. Loading the final building inventory containing guids...")
local_bldg_dataset = Dataset.from_file(
    "outputs/montpelier_building_inventory.shp", 
    data_type="ergo:buildingInventoryVer7"
)

print("4. Initializing and configuring the structural damage analysis model...")
bldg_dmg = BuildingStructuralDamage(client)
bldg_dmg.set_input_dataset("buildings", local_bldg_dataset)
bldg_dmg.set_input_dataset("dfr3_mapping_set", mapping_set)

# Pass the cloud hazard object
bldg_dmg.set_input_hazard("hazard", cloud_flood) 

bldg_dmg.set_parameter("hazard_type", "flood")
bldg_dmg.set_parameter("result_name", f"{OUT_DIR}/montpelier_bldg_flood_dmg")
bldg_dmg.set_parameter("num_cpu", 4)  # 4 CPUs full throttle

print("5. Starting the building structural damage analysis...")
bldg_dmg.run_analysis()
# print("🎉 Building structural damage analysis finally completed!")

1. Temporarily uploading local flood raster to your IN-CORE personal space (this only takes a few seconds)...
Upload successful! Cloud Dataset ID: 6a107a7cf8ba61644c24fd79
2. Reconstructing the hazard object using the cloud ID...
3. Loading the final building inventory containing guids...
4. Initializing and configuring the structural damage analysis model...
5. Starting the building structural damage analysis...


True

In [15]:
# Retrieve result dataset
building_dmg_result = bldg_dmg.get_output_dataset('ds_result')

In [16]:
# Convert dataset to Pandas DataFrame
bdmg_df = building_dmg_result.get_dataframe_from_csv(low_memory=False)

# Display top 5 rows of output data
bdmg_df.head()

,guid
0,f34dfef7-9bf9-43c4-aa1b-a5eb1b53c376
1,a966c757-db72-490b-84ca-3d87f7564935
2,9f5974c4-00ad-4230-ba41-346a2ccda1ce
3,246fd9a5-6ba5-4d7c-bb64-51776b7c627b
4,0fa30543-49c2-4ff1-a3ab-4bf4931c477b


In [17]:
# import matplotlib.pyplot as plt

# # Join damage results back to geometry for plotting
# bldg_plot = gpd.read_file(BLDG_SHP)
# bldg_plot = bldg_plot.merge(bdmg_df[["guid", "DS_0", "DS_1", "DS_2", "DS_3", "haz_expose"]],
#                              on="guid", how="left")

# fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# # Left: probability of DS_3 (extensive/complete damage)
# bldg_plot.plot(column="DS_3", cmap="Reds", legend=True,
#                legend_kwds={"label": "P(DS_3 — extensive/complete)"},
#                ax=axes[0], markersize=4)
# axes[0].set_title("Probability of Extensive/Complete Damage (DS3)")
# axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")

# # Right: hazard exposure
# colors = bldg_plot["haz_expose"].map({"yes": "red", "no": "steelblue"}).fillna("grey")
# bldg_plot.plot(color=colors, ax=axes[1], markersize=4)
# axes[1].set_title("Flood Exposure (red = exposed)")
# axes[1].set_xlabel("Longitude")

# plt.tight_layout()
# plt.savefig(f"{OUT_DIR}/building_damage_map.png", dpi=150, bbox_inches="tight")
# plt.show()
# print(f"Saved: {OUT_DIR}/building_damage_map.png")

### 3.5 Monte Carlo failure probability

In [ ]:
mc = MonteCarloFailureProbability(client)

mc.set_input_dataset("damage", building_dmg_result)
mc.set_parameter("num_cpu",             4)
mc.set_parameter("num_samples",         10000)
mc.set_parameter("damage_interval_keys", ["DS_0", "DS_1", "DS_2", "DS_3"])
mc.set_parameter("failure_state_keys",   ["DS_1", "DS_2", "DS_3"])
mc.set_parameter("result_name",          "montpelier_mc_failure")

mc.run_analysis()

mc_result = mc.get_output_dataset("failure_probability")
mc_df = mc_result.get_dataframe_from_csv(low_memory=False)

print(f"Monte Carlo complete  ({mc_df.shape[0]} buildings, 10,000 samples)")
print(f"Mean failure probability: {mc_df['failure_probability'].mean():.3f}")
mc_df.head()

---
## Part 4 — Housing Unit Allocation

The Housing Unit Allocation (HUA) links Census household demographics to individual buildings,
which is required for the Population Dislocation analysis.

HUA requires three inputs:
1. **Housing unit inventory** (`incore:housingUnitInventory`) — Census household records at the block level
2. **Address point inventory** (`incore:addressPoint`) — every deliverable address in the study area
3. **Building inventory** — our NSI shapefile

These first two must be built for Washington County, VT. The cells below fetch what can be
obtained automatically; the address points require a one-time manual download from VCGI.

### 4.1 Fetch Washington County VT block group demographics (for Population Dislocation)

In [ ]:
from pyincore_data.censusutil import CensusUtil

# Washington County, VT FIPS = 50023
state_counties = ["50023"]

print("Fetching 2010 Census block group data for Washington County VT ...")
blockgroup_df, bgmap, bg_outdataset = CensusUtil.get_blockgroupdata_for_dislocation(
    state_counties,
    out_csv=True,
    out_shapefile=False,
    out_html=False
)

print(f"Block groups fetched: {len(blockgroup_df)}")
print(f"Columns: {list(blockgroup_df.columns)}")
blockgroup_df.head()

### 4.2 Build housing unit inventory from Census ACS data

The housing unit inventory records one row per household with fields required by
pyIncore's HousingUnitAllocation: `huid`, `tractid`, `bgid`, `blockid`, `numprec` (number of people),
`ownershp`, `race`, `hispan`, `vacancy`, `gqtype`.

These come from Census IPUMS microdata at the block level. The cell below fetches
2010 Decennial Census block-level housing counts for Washington County, VT and
formats them into the pyIncore schema.

In [ ]:
# Fetch 2010 Census housing unit counts by block for Washington County VT
# Census API variables:
#   H001001 = total housing units
#   H003002 = occupied units
#   H003003 = vacant units
#   P001001 = total population

census_url = (
    "https://api.census.gov/data/2010/dec/sf1"
    "?get=GEO_ID,H001001,H003002,H003003,P001001"
    "&for=block:*"
    "&in=state:50%20county:023"   # VT (50) + Washington County (023)
)
resp = requests.get(census_url, timeout=30)
resp.raise_for_status()
rows = resp.json()
headers, data = rows[0], rows[1:]
census_df = pd.DataFrame(data, columns=headers)
for col in ["H001001", "H003002", "H003003", "P001001"]:
    census_df[col] = pd.to_numeric(census_df[col])

# Build blockid (15-digit FIPS: state+county+tract+block)
census_df["blockid"] = (
    census_df["state"] + census_df["county"] +
    census_df["tract"] + census_df["block"]
)
census_df["tractid"] = census_df["state"] + census_df["county"] + census_df["tract"]
census_df["bgid"]    = census_df["blockid"].str[:12]  # block group = first 12 digits

print(f"Census blocks: {len(census_df)}")
print(f"Total housing units in Washington County: {census_df['H001001'].sum():,}")
census_df.head()

In [ ]:
# Expand one row per housing unit (up to occupied count per block)
# Each housing unit gets a unique huid and a random population/tenure draw
# This is a simplified approximation — for full PUMS microdata use IPUMS

rng = np.random.default_rng(seed=42)
hu_rows = []
huid_counter = 1

for _, row in census_df.iterrows():
    n_occupied = int(row["H003002"])
    n_vacant   = int(row["H003003"])
    population = int(row["P001001"])
    
    # Approximate household size (people per occupied unit)
    avg_hh_size = population / n_occupied if n_occupied > 0 else 0
    
    for _ in range(n_occupied):
        hu_rows.append({
            "huid":     f"H{row['blockid']}{huid_counter:04d}",
            "blockid":  row["blockid"],
            "tractid":  row["tractid"],
            "bgid":     row["bgid"],
            "numprec":  max(1, round(rng.normal(avg_hh_size, 1))),
            "ownershp": int(rng.choice([1, 2], p=[0.62, 0.38])),  # VT ownership rate ~62%
            "race":     int(rng.choice([1,2,3,4,5], p=[0.93,0.02,0.01,0.02,0.02])),  # VT demographics
            "hispan":   int(rng.choice([0, 1], p=[0.985, 0.015])),
            "vacancy":  0,
            "gqtype":   0,
        })
        huid_counter += 1
    
    for _ in range(n_vacant):
        hu_rows.append({
            "huid": f"H{row['blockid']}{huid_counter:04d}",
            "blockid": row["blockid"], "tractid": row["tractid"], "bgid": row["bgid"],
            "numprec": 0, "ownershp": 0, "race": 0, "hispan": 0, "vacancy": 1, "gqtype": 0,
        })
        huid_counter += 1

hu_df = pd.DataFrame(hu_rows)
hu_path = f"{OUT_DIR}/washington_county_vt_housing_units.csv"
hu_df.to_csv(hu_path, index=False)

print(f"Housing unit inventory: {len(hu_df):,} records")
print(f"Saved: {hu_path}")
hu_df.head()

### 4.3 Address point inventory

Address points link housing units to buildings.  
Vermont E911 address points are available free from VCGI:

**Download:** https://geodata.vermont.gov/datasets/e911-site-locations/explore  
→ Export as CSV → filter to `TOWNNAME = 'Montpelier'` → save as `outputs/montpelier_address_points.csv`

The cell below checks for the file and formats it if present.

In [ ]:
AP_PATH = f"{OUT_DIR}/montpelier_address_points.csv"

if not os.path.exists(AP_PATH):
    print("Address point file not found.")
    print(f"Download Vermont E911 data from VCGI and save to: {AP_PATH}")
    print("Required columns: OBJECTID (or unique ID), LATITUDE, LONGITUDE, ADDNUM, PREFIXDIR, STREETNAME, SUFFIXDIR, STREETSUF, PLACENAME")
    print("\nSkipping HUA — proceeding to Population Dislocation with building damage only.")
    HUA_AVAILABLE = False
else:
    ap_df = pd.read_csv(AP_PATH)
    # Rename to pyIncore address point schema
    ap_df = ap_df.rename(columns={
        "OBJECTID": "addrptid",
        "LATITUDE": "y",
        "LONGITUDE": "x",
    })
    ap_gdf = gpd.GeoDataFrame(
        ap_df,
        geometry=gpd.points_from_xy(ap_df["x"], ap_df["y"]),
        crs="EPSG:4326"
    )
    ap_shp = f"{OUT_DIR}/montpelier_address_points"
    ap_gdf.to_file(ap_shp, driver="ESRI Shapefile")
    print(f"Address points loaded: {len(ap_gdf):,}")
    HUA_AVAILABLE = True

### 4.4 Run Housing Unit Allocation (if address points are available)

In [ ]:
if HUA_AVAILABLE:
    housing_unit_inv  = Dataset.from_file(hu_path, data_type="incore:housingUnitInventory")
    address_point_inv = Dataset.from_file(f"{ap_shp}/montpelier_address_points.shp",
                                          data_type="incore:addressPoint")

    hua = HousingUnitAllocation(client)
    hua.load_remote_input_dataset("housing_unit_inventory", housing_unit_inv)
    hua.load_remote_input_dataset("address_point_inventory", address_point_inv)
    hua.set_input_dataset("buildings", buildings)
    hua.set_parameter("result_name", "montpelier_HUA")
    hua.set_parameter("seed",        1238)
    hua.set_parameter("iterations",  1)
    hua.run_analysis()

    hua_result = hua.get_output_dataset("result")
    hua_df = hua_result.get_dataframe_from_csv(low_memory=False)
    print(f"HUA complete: {len(hua_df):,} household records")
    hua_df.head()
else:
    hua_result = None
    hua_df = None
    print("HUA skipped — address points not available.")

---
## Part 5 — Population Dislocation

Population dislocation estimates households forced to leave their pre-event residence
due to flood damage. It combines building damage states, household characteristics,
and neighborhood demographics.

### 5.1 Load value loss parameters and run dislocation

In [ ]:
if hua_result is not None:
    # Value loss parameters (DS 0–3) — same dataset as Lumberton testbed
    value_loss_id = "60354810e379f22e16560dbd"

    # Block group data (fetched in 4.1)
    bg_data = Dataset.from_file(
        "program_name/program_name_geo_name.csv",
        data_type="incore:blockGroupData"
    )

    pop_dis = PopulationDislocation(client)
    pop_dis.set_input_dataset("block_group_data",      bg_data)
    pop_dis.load_remote_input_dataset("value_loss_param", value_loss_id)
    pop_dis.set_input_dataset("building_dmg",          building_dmg_result)
    pop_dis.set_input_dataset("housing_unit_allocation", hua_result)
    pop_dis.set_parameter("result_name", "montpelier_pop_dislocation")
    pop_dis.set_parameter("seed", 1111)

    pop_dis.run_analysis()

    pd_result = pop_dis.get_output_dataset("result")
    pd_df = pd_result.get_dataframe_from_csv(low_memory=False)

    n_dislocated = pd_df["dislocated"].sum()
    print(f"Population dislocation complete")
    print(f"Total households analysed : {len(pd_df):,}")
    print(f"Estimated dislocated      : {int(n_dislocated):,} ({100*n_dislocated/len(pd_df):.1f}%)")
    pd_df.tail()
else:
    print("Population dislocation skipped — HUA results not available.")
    print("Complete Part 4 (obtain E911 address points) to enable this analysis.")

### 5.2 Summary table — dislocation by race/ethnicity

In [ ]:
if hua_result is not None:
    RACE_LABELS = {
        1: "White alone, Not Hispanic",
        2: "Black alone, Not Hispanic",
        3: "Am. Indian / Alaska Native",
        4: "Asian alone, Not Hispanic",
        5: "Other Race, Not Hispanic",
        6: "Any Race, Hispanic",
    }
    pd_df["race_label"] = pd_df["race"].map(RACE_LABELS).fillna("Unknown")

    summary = pd_df.groupby("race_label").agg(
        total_households=("dislocated", "count"),
        dislocated=("dislocated", "sum")
    )
    summary["pct_dislocated"] = (summary["dislocated"] / summary["total_households"] * 100).round(1)
    print("Population dislocation by race/ethnicity — Montpelier VT, July 2023 flood")
    print(summary.to_string())

---
## Output files

| File | Description |
|------|-------------|
| `outputs/montpelier_flood_depth_2023.tif` | Flood depth raster (ft above ground, EPSG:3857) |
| `outputs/montpelier_building_inventory/` | NSI building inventory shapefile |
| `montpelier_bldg_flood_dmg_ds_result.csv` | Building damage state probabilities |
| `montpelier_mc_failure_failure_probability.csv` | Monte Carlo failure probabilities |
| `outputs/building_damage_map.png` | Spatial damage visualization |
| `outputs/washington_county_vt_housing_units.csv` | Synthetic housing unit inventory |
| `montpelier_HUA_result.csv` | Housing unit allocation (if address points available) |
| `montpelier_pop_dislocation_result.csv` | Population dislocation (if HUA complete) |

**To enable the full pipeline (Parts 4–5):**
1. Download Vermont E911 address points from [VCGI](https://geodata.vermont.gov/datasets/e911-site-locations/explore)
2. Filter to `TOWNNAME = 'Montpelier'`
3. Export as CSV → save as `outputs/montpelier_address_points.csv`
4. Re-run cells 4.3 onwards